# ElementalTask-RML
## Notebook 01 — Emergence Order Monitoring

This notebook adds a lightweight Residue Manifold Learning (RML) / CGCS-inspired monitoring layer to **ElementalTask**.

Goal:

- load ElementalTask checkpoint-evaluation outputs,
- compute first-emergence checkpoint per task,
- rank tasks by emergence order,
- check simple ordering constraints such as atomic tasks emerging before compositional tasks,
- produce an early-warning view of constraint-score drift.

Pipeline:

`checkpoint → emergence rank → function-vector similarity → constraint-score drift`

**Emergence ≠ magic. Monitor constraints. 📐**

## 0. Notes

This notebook is intentionally conservative.

It does **not** claim a new proof or replacement for ElementalTask. It starts with one reproducible monitoring question:

> If emergence ordering is partially stable, can ordering violations flag capability drift before training completes?

Expected repo paths from upstream README:

- `dataset/simple.csv`
- `dataset/compositional.csv`
- `dataset/math_expressions.csv`
- `results/`
- `output/`
- `plots/`
- `function_vecs/`
- `scripts/trajectory_analysis/predict_compositional_from_components.py`

RML fork outputs from this notebook:

- `plots_rml/01_emergence_order_monitor.png`
- `plots_rml/01_cgcs_by_checkpoint.png`
- `docs_rml/01_emergence_order_monitor.md`
- `results_rml/01_emergence_order_table.csv`
- `results_rml/01_constraint_scores.csv`

In [ ]:
# 1. Imports + repo setup
from pathlib import Path
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.stats import spearmanr
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

warnings.filterwarnings("ignore", category=FutureWarning)

# Find repo root whether running from root, notebooks_rml/, or Colab.
def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    markers = ["dataset", "tasks", "function_vecs", "scripts"]
    for p in [start] + list(start.parents):
        if sum((p / m).exists() for m in markers) >= 2:
            return p
    return start

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

PLOTS_DIR = REPO_ROOT / "plots_rml"
DOCS_DIR = REPO_ROOT / "docs_rml"
RESULTS_DIR = REPO_ROOT / "results_rml"
for d in [PLOTS_DIR, DOCS_DIR, RESULTS_DIR]:
    d.mkdir(exist_ok=True, parents=True)

print(f"Repo root: {REPO_ROOT}")
print(f"Plots:    {PLOTS_DIR}")
print(f"Docs:     {DOCS_DIR}")
print(f"Results:  {RESULTS_DIR}")

## 2. Discover available evaluation CSVs

The upstream README mentions `results/`, `output/`, and plot commands such as:

```bash
python analysis/plotting.py --csv_path output/olmo2_ckpt_interp_results.csv --plot_type curves
```

This notebook searches likely folders and lets you choose the first detected checkpoint-evaluation CSV. If no suitable CSV exists yet, it creates a tiny synthetic demo dataset so the notebook still documents the intended analysis flow.

In [ ]:
# 2. Discover candidate CSV files
SEARCH_DIRS = ["results", "output", "outputs", "data", "results_rml"]
CSV_CANDIDATES = []

for dirname in SEARCH_DIRS:
    root = REPO_ROOT / dirname
    if root.exists():
        CSV_CANDIDATES.extend(sorted(root.rglob("*.csv")))

print(f"Found {len(CSV_CANDIDATES)} CSV files in likely output directories.")
for i, path in enumerate(CSV_CANDIDATES[:25]):
    print(f"[{i:02d}] {path.relative_to(REPO_ROOT)}")
if len(CSV_CANDIDATES) > 25:
    print("...")

In [ ]:
# 3. Helper functions for flexible checkpoint/task/score column detection
CHECKPOINT_CANDIDATES = ["checkpoint", "ckpt", "step", "global_step", "checkpoint_step", "tokens", "token_count"]
TASK_CANDIDATES = ["task", "task_name", "name", "dataset", "category", "subtask", "task_type"]
SCORE_CANDIDATES = ["accuracy", "acc", "exact_match", "em", "score", "mean_accuracy", "correct", "performance"]

def normalize_colname(c):
    return re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_")

def find_column(df, candidates):
    norm = {normalize_colname(c): c for c in df.columns}
    for cand in candidates:
        if cand in norm:
            return norm[cand]
    for cand in candidates:
        for nc, original in norm.items():
            if cand in nc:
                return original
    return None

def parse_checkpoint_value(x):
    """Convert checkpoint labels such as step10000-tokens42B to sortable numeric values when possible."""
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x)
    m = re.search(r"step[_-]?(\d+)", s, flags=re.I)
    if m:
        return float(m.group(1))
    m = re.search(r"tokens?[_-]?(\d+(?:\.\d+)?)([kmbt])?", s, flags=re.I)
    if m:
        val = float(m.group(1))
        mult = {None: 1, "k": 1e3, "m": 1e6, "b": 1e9, "t": 1e12}.get((m.group(2) or "").lower(), 1)
        return val * mult
    m = re.search(r"\d+(?:\.\d+)?", s)
    if m:
        return float(m.group(0))
    return np.nan

def standardize_eval_df(df):
    """Return a standardized dataframe with checkpoint, checkpoint_order, task, score."""
    checkpoint_col = find_column(df, CHECKPOINT_CANDIDATES)
    task_col = find_column(df, TASK_CANDIDATES)
    score_col = find_column(df, SCORE_CANDIDATES)

    if checkpoint_col is None or task_col is None or score_col is None:
        raise ValueError(
            "Could not detect required columns. "
            f"checkpoint={checkpoint_col}, task={task_col}, score={score_col}. "
            f"Columns: {list(df.columns)}"
        )

    out = df[[checkpoint_col, task_col, score_col]].copy()
    out.columns = ["checkpoint", "task", "score"]
    out["checkpoint_order"] = out["checkpoint"].apply(parse_checkpoint_value)

    if out["checkpoint_order"].isna().all():
        order_map = {v: i for i, v in enumerate(pd.unique(out["checkpoint"]), start=1)}
        out["checkpoint_order"] = out["checkpoint"].map(order_map).astype(float)

    out["score"] = pd.to_numeric(out["score"], errors="coerce")
    out = out.dropna(subset=["task", "score", "checkpoint_order"])
    out = out.sort_values(["checkpoint_order", "task"]).reset_index(drop=True)
    return out

In [ ]:
# 4. Load first likely evaluation CSV, or use synthetic demonstration data if none is available.
# You can manually set CSV_PATH to a specific file, for example:
# CSV_PATH = REPO_ROOT / "output" / "olmo2_ckpt_interp_results.csv"
CSV_PATH = None

loaded = None
load_errors = []

candidate_paths = [Path(CSV_PATH)] if CSV_PATH else CSV_CANDIDATES
for path in candidate_paths:
    try:
        raw = pd.read_csv(path)
        eval_df = standardize_eval_df(raw)
        loaded = path
        break
    except Exception as e:
        load_errors.append((str(path), str(e)))

if loaded is None:
    print("No compatible evaluation CSV found. Using synthetic demo data for notebook scaffolding.")
    demo_rows = []
    tasks = [
        ("simple:copying", 1),
        ("simple:uppercase", 2),
        ("simple:first_letter", 3),
        ("math:arithmetic", 4),
        ("compositional:copy_then_uppercase", 5),
        ("compositional:first_letter_then_uppercase", 6),
    ]
    checkpoints = [1000, 3000, 6000, 10000, 20000, 50000, 100000]
    rng = np.random.default_rng(42)
    for task, center in tasks:
        for i, ckpt in enumerate(checkpoints):
            score = 1 / (1 + np.exp(-(i - center) * 1.25))
            score += rng.normal(0, 0.035)
            demo_rows.append({"checkpoint": f"step{ckpt}", "task": task, "score": np.clip(score, 0, 1), "checkpoint_order": ckpt})
    eval_df = pd.DataFrame(demo_rows)
    loaded = "synthetic_demo"
else:
    print(f"Loaded: {Path(loaded).relative_to(REPO_ROOT)}")

print(eval_df.head())
print(eval_df.describe(include="all"))

## 3. Compute first-emergence checkpoint

For each task, define emergence as the first checkpoint where score exceeds a threshold.

Default threshold: `0.50`.

You can tune this threshold to match ElementalTask paper settings or task-specific evaluation choices.

In [ ]:
# 5. First-emergence checkpoint per task
EMERGENCE_THRESHOLD = 0.50

def compute_emergence_table(df, threshold=0.50):
    rows = []
    for task, g in df.sort_values("checkpoint_order").groupby("task"):
        emerged = g[g["score"] >= threshold]
        if len(emerged):
            first = emerged.iloc[0]
            rows.append({
                "task": task,
                "emerged": True,
                "emergence_checkpoint": first["checkpoint"],
                "emergence_order": float(first["checkpoint_order"]),
                "emergence_score": float(first["score"]),
                "max_score": float(g["score"].max()),
            })
        else:
            rows.append({
                "task": task,
                "emerged": False,
                "emergence_checkpoint": None,
                "emergence_order": np.inf,
                "emergence_score": np.nan,
                "max_score": float(g["score"].max()),
            })
    out = pd.DataFrame(rows)
    out["emergence_rank"] = out["emergence_order"].rank(method="dense", na_option="bottom").astype(int)
    return out.sort_values(["emergence_order", "task"]).reset_index(drop=True)

emergence = compute_emergence_table(eval_df, EMERGENCE_THRESHOLD)
emergence.to_csv(RESULTS_DIR / "01_emergence_order_table.csv", index=False)
emergence

## 4. Infer task groups

This first notebook uses conservative string heuristics:

- task names containing `simple`, `copy`, `uppercase`, `lowercase`, `first_letter`, or `atomic` are treated as atomic/simple candidates;
- task names containing `compositional`, `compose`, `then`, `chain`, or `multi` are treated as compositional candidates;
- task names containing `math` or `arithmetic` are treated as math candidates.

Later notebooks can replace this with an explicit constraint graph derived from task metadata.

In [ ]:
# 6. Task group heuristics
ATOMIC_PAT = re.compile(r"simple|atomic|copy|uppercase|lowercase|first[_ -]?letter|last[_ -]?letter|reverse|token", re.I)
COMPOSITE_PAT = re.compile(r"compositional|composition|compose|then|chain|multi|combined", re.I)
MATH_PAT = re.compile(r"math|arithmetic|add|subtract|multiply|divide|expression", re.I)

def infer_task_group(task):
    s = str(task)
    if COMPOSITE_PAT.search(s):
        return "compositional"
    if MATH_PAT.search(s):
        return "math"
    if ATOMIC_PAT.search(s):
        return "atomic"
    return "unknown"

emergence["task_group"] = emergence["task"].apply(infer_task_group)
emergence.to_csv(RESULTS_DIR / "01_emergence_order_table.csv", index=False)
emergence[["task", "task_group", "emerged", "emergence_checkpoint", "emergence_rank", "max_score"]]

## 5. Minimal CGCS ordering score

Initial constraint:

> atomic/simple tasks should emerge before compositional tasks.

Minimal score:

\[
CGCS = 1 - \frac{\text{ordering violations}}{\text{total constraints}}
\]

This is deliberately simple. It is a starter monitor, not a final metric.

In [ ]:
# 7. Minimal pairwise ordering constraints and CGCS score

def build_pairwise_constraints(emergence_df):
    atomic = emergence_df[emergence_df["task_group"].eq("atomic")]
    composite = emergence_df[emergence_df["task_group"].eq("compositional")]

    rows = []
    for _, a in atomic.iterrows():
        for _, c in composite.iterrows():
            valid = a["emergence_order"] <= c["emergence_order"]
            rows.append({
                "prerequisite_task": a["task"],
                "target_task": c["task"],
                "constraint": "atomic_before_compositional",
                "prereq_order": a["emergence_order"],
                "target_order": c["emergence_order"],
                "valid": bool(valid),
                "violation": bool(not valid),
            })
    return pd.DataFrame(rows)

constraints = build_pairwise_constraints(emergence)

if len(constraints):
    violations = int(constraints["violation"].sum())
    total_constraints = int(len(constraints))
    cgcs = 1 - violations / total_constraints
else:
    violations = 0
    total_constraints = 0
    cgcs = np.nan

summary = pd.DataFrame([{
    "threshold": EMERGENCE_THRESHOLD,
    "total_constraints": total_constraints,
    "ordering_violations": violations,
    "cgcs": cgcs,
    "source": str(loaded),
}])

constraints.to_csv(RESULTS_DIR / "01_pairwise_constraints.csv", index=False)
summary.to_csv(RESULTS_DIR / "01_constraint_scores.csv", index=False)

print(summary.to_string(index=False))
constraints.head(20)

## 6. Checkpoint-level drift score

For each checkpoint, compare current task ordering by score with final task ordering by score.

This gives a simple monitoring view:

- high rank correlation: current checkpoint resembles final ordering,
- low rank correlation: current capability ordering is still unstable.

This is a pragmatic early-warning diagnostic, not a claim about causal model internals.

In [ ]:
# 8. Rank-order stability over checkpoints

def rank_stability_by_checkpoint(df):
    pivot = df.pivot_table(index="checkpoint_order", columns="task", values="score", aggfunc="mean").sort_index()
    pivot_filled = pivot.interpolate(axis=0).ffill().bfill()
    final_scores = pivot_filled.iloc[-1]

    rows = []
    for ckpt_order, row in pivot_filled.iterrows():
        common = row.dropna().index.intersection(final_scores.dropna().index)
        if len(common) >= 2:
            if SCIPY_AVAILABLE:
                rho, p = spearmanr(row[common], final_scores[common])
            else:
                rho = pd.Series(row[common]).rank().corr(pd.Series(final_scores[common]).rank(), method="pearson")
                p = np.nan
        else:
            rho, p = np.nan, np.nan
        rows.append({"checkpoint_order": ckpt_order, "spearman_to_final": rho, "p_value": p, "n_tasks": len(common)})
    return pd.DataFrame(rows)

stability = rank_stability_by_checkpoint(eval_df)
stability.to_csv(RESULTS_DIR / "01_rank_stability_by_checkpoint.csv", index=False)
stability.head()

## 7. Visualizations

The first figure shows task trajectories and emergence threshold.

The second figure shows rank-order stability versus the final checkpoint.

In [ ]:
# 9. Plot task trajectories
plt.figure(figsize=(12, 7))
for task, g in eval_df.groupby("task"):
    plt.plot(g["checkpoint_order"], g["score"], marker="o", linewidth=1.5, label=str(task)[:40])
plt.axhline(EMERGENCE_THRESHOLD, linestyle="--", linewidth=1, label=f"threshold={EMERGENCE_THRESHOLD}")
plt.xlabel("Checkpoint order")
plt.ylabel("Score")
plt.title("ElementalTask-RML: Emergence Trajectories")
plt.legend(loc="best", fontsize=8)
plt.tight_layout()
fig1 = PLOTS_DIR / "01_emergence_order_monitor.png"
plt.savefig(fig1, dpi=180)
plt.show()
print(f"Saved {fig1.relative_to(REPO_ROOT)}")

In [ ]:
# 10. Plot rank stability to final checkpoint
plt.figure(figsize=(10, 5))
plt.plot(stability["checkpoint_order"], stability["spearman_to_final"], marker="o", linewidth=2)
plt.ylim(-1.05, 1.05)
plt.axhline(0, linestyle="--", linewidth=1)
plt.xlabel("Checkpoint order")
plt.ylabel("Spearman rank correlation to final")
plt.title("ElementalTask-RML: Rank-Order Stability Monitor")
plt.tight_layout()
fig2 = PLOTS_DIR / "01_rank_stability_monitor.png"
plt.savefig(fig2, dpi=180)
plt.show()
print(f"Saved {fig2.relative_to(REPO_ROOT)}")

In [ ]:
# 11. Plot minimal CGCS summary
if total_constraints > 0:
    labels = ["valid", "violations"]
    values = [total_constraints - violations, violations]
    plt.figure(figsize=(7, 5))
    plt.bar(labels, values)
    plt.ylabel("Pairwise constraints")
    plt.title(f"Minimal CGCS = {cgcs:.3f}")
    plt.tight_layout()
    fig3 = PLOTS_DIR / "01_cgcs_by_checkpoint.png"
    plt.savefig(fig3, dpi=180)
    plt.show()
    print(f"Saved {fig3.relative_to(REPO_ROOT)}")
else:
    print("No atomic-to-compositional constraints detected; CGCS bar plot skipped.")

## 8. Optional function-vector hook

The upstream README says function-vector extraction lives in `function_vecs/` and exposes:

```python
from function_vecs.extract_function_vecs import extract_function_vector_simple
```

This notebook does not run function-vector extraction by default because model loading can be expensive. The cell below is a safe hook for later use.

In [ ]:
# 12. Optional FV extraction hook; keep disabled until dependencies/model paths are ready.
RUN_FV_EXAMPLE = False

if RUN_FV_EXAMPLE:
    from function_vecs.extract_function_vecs import extract_function_vector_simple

    fv = extract_function_vector_simple(
        task_name="simple_icl",
        model_name="distilgpt2",
        num_samples=5,
        device="cpu",
    )
    print(f"Function vector shape: {fv.function_vec.shape}")
    print(f"Task name: {fv.task_name}")
else:
    print("FV example skipped. Set RUN_FV_EXAMPLE = True after dependencies/model paths are ready.")

## 9. Export markdown report

This writes a small report that can be committed under `docs_rml/`.

In [ ]:
# 13. Export lightweight markdown report
cgcs_text = "N/A" if np.isnan(cgcs) else f"{cgcs:.6f}"
report = f"""# Notebook 01 — Emergence Order Monitoring

Source: `{loaded}`

Emergence threshold: `{EMERGENCE_THRESHOLD}`

## Summary

- Tasks analyzed: `{eval_df['task'].nunique()}`
- Checkpoints analyzed: `{eval_df['checkpoint_order'].nunique()}`
- Pairwise atomic-to-compositional constraints: `{total_constraints}`
- Ordering violations: `{violations}`
- Minimal CGCS: `{cgcs_text}`

## Pipeline

`checkpoint -> emergence rank -> function-vector similarity -> constraint-score drift`

## Generated artifacts

- `results_rml/01_emergence_order_table.csv`
- `results_rml/01_pairwise_constraints.csv`
- `results_rml/01_constraint_scores.csv`
- `results_rml/01_rank_stability_by_checkpoint.csv`
- `plots_rml/01_emergence_order_monitor.png`
- `plots_rml/01_rank_stability_monitor.png`
- `plots_rml/01_cgcs_by_checkpoint.png`

## Interpretation

This notebook tests whether checkpoint-level emergence ordering can serve as a lightweight training monitor.

A useful next step is replacing pairwise string-heuristic constraints with an explicit task graph derived from ElementalTask task metadata.

**Emergence ≠ magic. Monitor constraints. 📐**
"""

report_path = DOCS_DIR / "01_emergence_order_monitor.md"
report_path.write_text(report, encoding="utf-8")
print(report_path.relative_to(REPO_ROOT))
print(report[:1000])

## 10. Optional Colab zip download

Uncomment the cell below in Colab to zip generated RML figures, docs, and CSV outputs.

In [ ]:
# # 14. Optional: zip generated RML artifacts and download in Colab
# import zipfile
#
# EXPORT_NAME = "elemental_task_rml_notebook01_artifacts.zip"
# EXPORT_PATH = REPO_ROOT / EXPORT_NAME
#
# include_dirs = [PLOTS_DIR, DOCS_DIR, RESULTS_DIR]
# with zipfile.ZipFile(EXPORT_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
#     for folder in include_dirs:
#         if folder.exists():
#             for path in folder.rglob("*"):
#                 if path.is_file():
#                     zf.write(path, path.relative_to(REPO_ROOT))
#
# print(f"Wrote {EXPORT_PATH}")
#
# try:
#     from google.colab import files
#     files.download(str(EXPORT_PATH))
# except Exception as e:
#     print("Not running in Colab or download unavailable.")
#     print(e)

## 11. Next steps

Recommended next notebooks:

1. `02_constraint_graph_from_task_metadata.ipynb`  
   Replace string heuristics with explicit task prerequisites.

2. `03_function_vector_similarity_drift.ipynb`  
   Compare FV similarity across tasks and checkpoints.

3. `04_rml_lane_visualization.ipynb`  
   Visualize emergence trajectories as RML-style lanes without changing upstream ElementalTask evaluation logic.